# Phase 1 — Baseline SFT (No Chain-of-Thought)
## NVIDIA Nemotron Reasoning Challenge

**Goal:** Get a real leaderboard score as fast as possible. Sets the floor we beat in Phase 2.

**What we do:**
1. Load Nemotron-3-Nano-30B (SSM/Mamba hybrid, ~3B active params via MoE)
2. Apply LoRA rank-32 adapter (max allowed by competition)
3. Train on raw `(prompt → \\boxed{answer})` pairs — **no reasoning chains**
4. Evaluate on held-out 20% split → per-category breakdown
5. Save adapter → zip → submit

---
### Architecture Notes (Important!)
- This is **NOT** a standard transformer — it uses Mamba SSM + MoE layers
- **No Unsloth** (Unsloth doesn't support Mamba)
- LoRA target modules are SSM-specific: `in_proj`, `out_proj`, `up_proj`, `down_proj`
- Model loads in `bfloat16` with CPU offloading for inactive experts
- Requires the `mamba_ssm` + `nvidia_cutlass_dsl` packages (Kaggle environment only)

## Cell 1 — Environment Setup

> **Must run first.** Adds the CUTLASS DSL path needed to import `mamba_ssm`.

In [23]:
import site
import os
import sys
import shutil

# Required for Nemotron-3-Nano: adds NVIDIA CUTLASS DSL to sys.path
CUTLASS_PATH = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"
site.addsitedir(CUTLASS_PATH)

# Suppress tokenizer parallelism warnings during training
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 1. Paths
read_only_triton = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton"
writable_dir = "/kaggle/working/python_packages"
writable_triton = os.path.join(writable_dir, "triton")

# 2. Copy the entire triton package to a writable directory if it doesn't exist
if not os.path.exists(writable_triton):
    os.makedirs(writable_dir, exist_ok=True)
    print("Copying triton package to a writable directory...")
    shutil.copytree(read_only_triton, writable_triton)
    print("Copy complete.")

# 3. Recursively add execute permissions to the bin directory
bin_dir = os.path.join(writable_triton, "backends", "nvidia", "bin")
if os.path.exists(bin_dir):
    for root, dirs, files in os.walk(bin_dir):
        for file in files:
            file_path = os.path.join(root, file)
            os.chmod(file_path, 0o755)
    print("Execute permissions granted to triton binaries.")

# 4. Prepend the writable directory to sys.path so Python imports this copy FIRST
if writable_dir not in sys.path:
    sys.path.insert(0, writable_dir)

print("Environment setup complete.")

Copying triton package to a writable directory...
Copy complete.
Execute permissions granted to triton binaries.
Environment setup complete.


## Cell 2 — Imports

In [24]:
import re
import json
import subprocess
from pathlib import Path

import polars as pl
import pandas as pd
import numpy as np
import torch

import kagglehub
import mamba_ssm  # noqa: F401  — must be imported before transformers for Nemotron

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, TaskType
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import Dataset

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}, {props.total_memory / 1e9:.1f} GB VRAM")

PyTorch version : 2.12.0.dev20260324+cu128
CUDA available  : True
  GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition, 102.0 GB VRAM


## Cell 3 — Configuration

All hyperparameters in one place. Edit here before running.

In [25]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR    = "/kaggle/input/nvidia-nemotron-3-reasoning-challenge"
TRAIN_PATH  = f"{DATA_DIR}/train.csv"
TEST_PATH   = f"{DATA_DIR}/test.csv"
OUTPUT_DIR  = "/kaggle/working"          # adapter saved here
ADAPTER_DIR = f"{OUTPUT_DIR}/phase1_adapter"

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_HANDLE = "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"

# ── LoRA (rank must be ≤ 32 per competition rules) ────────────────────────────
LORA_RANK    = 32
LORA_ALPHA   = 64          # 2× rank is a good default
LORA_DROPOUT = 0.05
# Mamba/SSM target modules (NOT q_proj/k_proj/v_proj — those are transformer-only)
LORA_TARGET_MODULES = r".*\.(in_proj|out_proj|up_proj|down_proj)$"

# ── Training ──────────────────────────────────────────────────────────────────
MAX_SEQ_LENGTH  = 512       # Phase 1: no CoT, prompts are short (~100 tokens)
                            # Phase 2 will use 4096 to accommodate reasoning chains
BATCH_SIZE      = 4         # Must be 1 on T4/P100 with this model size
GRAD_ACCUM      = 4        # Effective batch size = 16
LEARNING_RATE   = 2e-4
NUM_EPOCHS      = 3
LR_SCHEDULER    = "cosine"
WARMUP_RATIO    = 0.05

# ── Eval split ────────────────────────────────────────────────────────────────
EVAL_SPLIT = 0.20           # 20% held out for local accuracy measurement
RANDOM_SEED = 42

# ── System prompt ─────────────────────────────────────────────────────────────
# Minimal — the puzzle prompt already contains all necessary context + examples.
# Don't over-engineer; the model already knows it's a reasoning task.
SYSTEM_PROMPT = (
    "You are a logical reasoning assistant. "
    "Analyze the given examples carefully to identify the hidden rule, "
    "then apply it to the new input. "
    "Place your final answer inside \\boxed{}."
)

print("Configuration:")
print(f"  LoRA rank       : {LORA_RANK}")
print(f"  Max seq length  : {MAX_SEQ_LENGTH}")
print(f"  Batch size      : {BATCH_SIZE} (effective: {BATCH_SIZE * GRAD_ACCUM})")
print(f"  Learning rate   : {LEARNING_RATE}")
print(f"  Epochs          : {NUM_EPOCHS}")
print(f"  Eval split      : {int(EVAL_SPLIT * 100)}%")

Configuration:
  LoRA rank       : 32
  Max seq length  : 512
  Batch size      : 4 (effective: 16)
  Learning rate   : 0.0002
  Epochs          : 3
  Eval split      : 20%


## Cell 4 — Load Data & Train/Eval Split

In [26]:
# Load full training data
df = pl.read_csv(TRAIN_PATH).to_pandas()
print(f"Loaded {len(df)} training examples.")

# Reproducible 80/20 split for local eval
np.random.seed(RANDOM_SEED)
shuffle_idx = np.random.permutation(len(df))
n_eval = int(len(df) * EVAL_SPLIT)

eval_df  = df.iloc[shuffle_idx[:n_eval]].reset_index(drop=True)
train_df = df.iloc[shuffle_idx[n_eval:]].reset_index(drop=True)

print(f"Train split : {len(train_df)} examples")
print(f"Eval split  : {len(eval_df)} examples")

Loaded 9500 training examples.
Train split : 7600 examples
Eval split  : 1900 examples


## Cell 5 — Format Training Data

Format: `system + user prompt → \boxed{answer}` (no reasoning chain in Phase 1).

The model learns to:
1. Recognize the `\boxed{}` output format (critical for scoring)
2. Associate prompt patterns with correct answer types
3. Fine-tune on the specific "Alice's Wonderland" domain

In [11]:
def format_example(prompt: str, answer: str) -> str:
    """
    Format a single (prompt, answer) pair into a training string.
    
    Uses a simple chat-style format compatible with Nemotron-3-Nano.
    The answer is wrapped in \\boxed{} as required by the evaluation metric.
    
    NOTE: In Phase 2, the assistant turn will contain CoT reasoning
    before the \\boxed{} answer. The structure is identical — only the
    assistant content changes.
    """
    return (
        f"<|system|>\n{SYSTEM_PROMPT}\n"
        f"<|user|>\n{prompt}\n"
        f"<|assistant|>\n\\boxed{{{answer}}}"
    )


# Apply formatting
train_df["text"] = [
    format_example(row["prompt"], row["answer"])
    for _, row in train_df.iterrows()
]

# Quick sanity check
print("=" * 60)
print("Sample formatted training example:")
print("=" * 60)
sample = train_df.iloc[0]
print(sample["text"])
print(f"\nChar length: {len(sample['text'])}")

Sample formatted training example:
<|system|>
You are a logical reasoning assistant. Analyze the given examples carefully to identify the hidden rule, then apply it to the new input. Place your final answer inside \boxed{}.
<|user|>
In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
00001000 -> 00100000
11010011 -> 01001111
01101110 -> 10111001
10110110 -> 11011010
10110010 -> 11001010
00101010 -> 10101000
10100110 -> 10011010
10101001 -> 10100110
11110111 -> 11011111

Now, determine the output for: 10011101
<|assistant|>
\boxed{01110110}

Char length: 718


## Cell 6 — Load Model & Tokenizer

Downloads Nemotron-3-Nano-30B from Kaggle model hub (~60GB, cached after first run).
Loads in `bfloat16` with `device_map="auto"` for CPU offloading of inactive expert layers.

In [12]:
print("Downloading model from Kaggle hub (cached after first run)...")
MODEL_PATH = kagglehub.model_download(MODEL_HANDLE)
print(f"Model path: {MODEL_PATH}")

# ── Load tokenizer ─────────────────────────────────────────────────────────────
print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
)

# Nemotron tokenizer may not have a pad token — set it to eos if so
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Vocab size       : {tokenizer.vocab_size}")
print(f"Pad token        : {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")
print(f"EOS token        : {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")

# ── Load model ─────────────────────────────────────────────────────────────────
print("\nLoading model (this may take a few minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",          # auto-distributes across GPU + CPU
    trust_remote_code=True,
    torch_dtype=torch.bfloat16, # native precision for Nemotron
    offload_folder="offload_dir",  # inactive MoE experts offload here
)

model.config.use_cache = False  # disable KV cache during training
print("Model loaded successfully.")

# Report VRAM usage
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        allocated = torch.cuda.memory_allocated(i) / 1e9
        reserved  = torch.cuda.memory_reserved(i) / 1e9
        print(f"  GPU {i} VRAM: {allocated:.2f} GB allocated / {reserved:.2f} GB reserved")

Model path: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1

Loading tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


Vocab size       : 131072
Pad token        : '<|im_end|>' (id=11)
EOS token        : '<|im_end|>' (id=11)

Loading model (this may take a few minutes)...


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

Model loaded successfully.
  GPU 0 VRAM: 63.17 GB allocated / 63.17 GB reserved


## Cell 7 — Apply LoRA Adapter

LoRA target modules for Mamba/SSM:
- `in_proj` / `out_proj` — state-space model projections
- `up_proj` / `down_proj` — MLP/FFN projections

**NOT** `q_proj`, `k_proj`, `v_proj` — those are transformer attention layers and
don't exist in Mamba SSM blocks.

In [13]:
print(f"Applying LoRA (rank={LORA_RANK}, alpha={LORA_ALPHA})...")

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,  # regex matching SSM module names
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Verify LoRA modules were applied correctly
lora_modules = [name for name, _ in model.named_modules() if "lora_" in name]
print(f"\nNumber of LoRA modules: {len(lora_modules)}")
if lora_modules:
    print(f"Sample LoRA module names:")
    for name in lora_modules[:5]:
        print(f"  {name}")

Applying LoRA (rank=32, alpha=64)...


/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 880,138,240 || all params: 32,458,075,584 || trainable%: 2.7116

Number of LoRA modules: 53820
Sample LoRA module names:
  base_model.model.backbone.layers.0.mixer.in_proj.lora_dropout
  base_model.model.backbone.layers.0.mixer.in_proj.lora_dropout.default
  base_model.model.backbone.layers.0.mixer.in_proj.lora_A
  base_model.model.backbone.layers.0.mixer.in_proj.lora_A.default
  base_model.model.backbone.layers.0.mixer.in_proj.lora_B


## Cell 8 — Prepare Dataset

Tokenize and convert to HuggingFace Dataset format for SFTTrainer.

In [14]:
# Convert to HuggingFace Dataset
train_dataset = Dataset.from_dict({"text": train_df["text"].tolist()})

# Tokenize dataset for standard Trainer
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=MAX_SEQ_LENGTH)

train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Data collator for causal LM
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print(f"Training dataset: {len(train_dataset)} examples")

# Verify tokenization of a sample
sample_text = train_df["text"].iloc[0]
sample_tokens = tokenizer(sample_text, return_tensors="pt")
print(f"Sample token count: {sample_tokens['input_ids'].shape[1]}")
print(f"  (MAX_SEQ_LENGTH = {MAX_SEQ_LENGTH})")

if sample_tokens['input_ids'].shape[1] > MAX_SEQ_LENGTH:
    print("⚠ WARNING: sample exceeds MAX_SEQ_LENGTH — it will be truncated!")
    print("  Consider increasing MAX_SEQ_LENGTH if this is frequent.")
else:
    print("✅ Sample fits within MAX_SEQ_LENGTH.")

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Training dataset: 7600 examples
Sample token count: 307
  (MAX_SEQ_LENGTH = 512)
✅ Sample fits within MAX_SEQ_LENGTH.


## Cell 9 — Train

Uses TRL's `SFTTrainer` which handles:
- Tokenization with padding/truncation
- Gradient checkpointing
- Logging

> **Expected training time on Kaggle T4/P100:** ~2–4 hours for 3 epochs on 7600 examples
> Monitor GPU utilization in the right sidebar. If it's consistently low (<60%), something is wrong.

In [27]:
training_args = TrainingArguments(
    # ── Output ────────────────────────────────────────────────────────────────
    output_dir=OUTPUT_DIR,
    # overwrite_output_dir=True,

    # ── Training schedule ─────────────────────────────────────────────────────
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_ratio=WARMUP_RATIO,

    # ── Memory optimization ───────────────────────────────────────────────────
    gradient_checkpointing=True,
    fp16=False,
    bf16=True,              # match model dtype
    dataloader_num_workers=0,

    # ── Sequence length ───────────────────────────────────────────────────────
    # max_seq_length=MAX_SEQ_LENGTH,
        
    # ── Logging ───────────────────────────────────────────────────────────────
    logging_steps=10,
    logging_dir=f"{OUTPUT_DIR}/logs",
    report_to="none",       # disable wandb/tensorboard on Kaggle free tier

    # ── Checkpoint saving ─────────────────────────────────────────────────────
    save_strategy="no",     # save only at the end (saves Kaggle storage quota)
    seed=RANDOM_SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

print("Starting training...")
print(f"  Steps per epoch : {len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)}")
print(f"  Total steps     : {len(train_dataset) * NUM_EPOCHS // (BATCH_SIZE * GRAD_ACCUM)}")
print()

train_result = trainer.train()

print("\n✅ Training complete!")
print(f"  Loss at end : {train_result.training_loss:.4f}")
print(f"  Runtime     : {train_result.metrics.get('train_runtime', 0) / 60:.1f} minutes")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Starting training...
  Steps per epoch : 475
  Total steps     : 1425



PermissionError: [Errno 13] Permission denied: '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell'

## Cell 10 — Local Evaluation

Run inference on the held-out 20% split.
Compute overall accuracy and per-category accuracy.

> **This tells us:**
> - Which categories the model already handles well (expect roman_numerals to be high)
> - Where Phase 2 CoT will have the most impact (expect binary + other to be lowest)
> - Whether the `\boxed{}` format is being followed consistently

In [ ]:
# ── Category classifier (same as Phase 0 EDA) ─────────────────────────────────
def classify_puzzle(prompt: str, answer: str) -> str:
    answer_s = str(answer).strip()
    if re.fullmatch(r'[01]{4,}', answer_s):
        return "binary"
    if any(kw in prompt.lower() for kw in ["binary", "bit", "xor", "shift", "bitwise"]):
        return "binary"
    if re.fullmatch(r'[IVXLCDM]+', answer_s):
        return "roman_numerals"
    if re.fullmatch(r'-?\d+', answer_s):
        return "integer_math"
    if re.fullmatch(r'-?\d+\.\d+', answer_s):
        return "decimal_math"
    words = answer_s.split()
    if len(words) >= 2 and all(re.fullmatch(r'[a-zA-Z]+', w) for w in words):
        return "word_sequence"
    if re.fullmatch(r'[a-zA-Z]+', answer_s):
        return "word_single"
    return "other"


# ── Answer extraction ─────────────────────────────────────────────────────────
def extract_boxed_answer(text: str) -> str | None:
    """
    Extract content from \\boxed{...} in generated text.
    Mirrors the competition metric's extraction logic.
    Falls back to last numeric value if no \\boxed{} found.
    """
    # Primary: find \boxed{...}
    match = re.search(r'\\boxed\{([^}]+)\}', text)
    if match:
        return match.group(1).strip()

    # Fallback 1: last line might be the answer
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    if lines:
        return lines[-1]

    return None


# ── Answer grading ────────────────────────────────────────────────────────────
def grade_answer(predicted: str | None, ground_truth: str) -> bool:
    """
    Grade a prediction as correct or incorrect.
    Mirrors competition metric: exact string match OR numerical tolerance.
    """
    if predicted is None:
        return False

    predicted = predicted.strip()
    ground_truth = ground_truth.strip()

    # Exact string match (covers binary, roman numerals, word sequences)
    if predicted == ground_truth:
        return True

    # Numerical tolerance match (covers decimal_math)
    try:
        pred_f = float(predicted)
        gt_f   = float(ground_truth)
        if abs(pred_f - gt_f) / max(abs(gt_f), 1.0) < 1e-3:
            return True
    except (ValueError, TypeError):
        pass

    return False


print("Evaluation helpers defined ✅")

In [ ]:
# ── Run inference on eval set ─────────────────────────────────────────────────
print(f"Running inference on {len(eval_df)} eval examples...")
print("(This takes a while — ~1–2 seconds per example on T4)")
print()

model.eval()

predictions = []
categories  = []
correct     = []

for i, (_, row) in enumerate(eval_df.iterrows()):
    # Build inference prompt (no answer — model must generate it)
    input_text = (
        f"<|system|>\n{SYSTEM_PROMPT}\n"
        f"<|user|>\n{row['prompt']}\n"
        f"<|assistant|>\n"
    )

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=64,          # Phase 1: answer only, very short
            do_sample=False,            # match evaluation (temperature=0)
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    generated_text = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # Extract and grade answer
    predicted = extract_boxed_answer(generated_text)
    category  = classify_puzzle(row["prompt"], row["answer"])
    is_correct = grade_answer(predicted, row["answer"])

    predictions.append(predicted)
    categories.append(category)
    correct.append(is_correct)

    # Progress every 50 examples
    if (i + 1) % 50 == 0 or (i + 1) == len(eval_df):
        acc_so_far = sum(correct) / len(correct)
        print(f"  [{i+1:4d}/{len(eval_df)}]  Running accuracy: {acc_so_far:.3f}")

print("\n✅ Inference complete.")

In [ ]:
# ── Compile results ───────────────────────────────────────────────────────────
eval_df["category"]  = categories
eval_df["predicted"] = predictions
eval_df["correct"]   = correct

overall_acc = sum(correct) / len(correct)
boxed_rate  = sum(1 for p in predictions if p is not None and re.search(r'\\boxed', str(p)) or True) / len(predictions)

print("=" * 60)
print("PHASE 1 LOCAL EVALUATION RESULTS")
print("=" * 60)
print(f"\nOverall accuracy : {overall_acc:.4f} ({sum(correct)}/{len(correct)})")
print()

# Per-category breakdown
print("Per-category accuracy:")
print(f"{'Category':<20} {'Correct':>7} {'Total':>7} {'Accuracy':>10}")
print("-" * 48)
for cat in sorted(eval_df["category"].unique()):
    cat_df = eval_df[eval_df["category"] == cat]
    cat_correct = cat_df["correct"].sum()
    cat_total   = len(cat_df)
    cat_acc     = cat_correct / cat_total if cat_total > 0 else 0
    print(f"{cat:<20} {cat_correct:>7} {cat_total:>7} {cat_acc:>10.4f}")

print("-" * 48)
print(f"{'OVERALL':<20} {sum(correct):>7} {len(correct):>7} {overall_acc:>10.4f}")

# ── Format compliance ─────────────────────────────────────────────────────────
no_boxed = sum(1 for p in predictions if p is None)
print(f"\nFormat compliance:")
print(f"  No \\boxed{{}} found  : {no_boxed} ({100*no_boxed/len(predictions):.1f}%)")
print(f"  Answer present    : {len(predictions) - no_boxed} ({100*(len(predictions)-no_boxed)/len(predictions):.1f}%)")

# ── Sample errors (for debugging) ────────────────────────────────────────────
errors = eval_df[~eval_df["correct"]].head(5)
print(f"\nSample incorrect predictions (first 5):")
for _, row in errors.iterrows():
    print(f"  Category : {row['category']}")
    print(f"  Expected : {row['answer']!r}")
    print(f"  Got      : {row['predicted']!r}")
    print()

In [ ]:
# Save eval results for later comparison with Phase 2
eval_results = {
    "phase": "phase1_baseline",
    "overall_accuracy": float(overall_acc),
    "n_correct": int(sum(correct)),
    "n_total": int(len(correct)),
    "per_category": {}
}

for cat in sorted(eval_df["category"].unique()):
    cat_df = eval_df[eval_df["category"] == cat]
    eval_results["per_category"][cat] = {
        "correct": int(cat_df["correct"].sum()),
        "total": int(len(cat_df)),
        "accuracy": float(cat_df["correct"].mean()),
    }

with open(f"{OUTPUT_DIR}/phase1_eval_results.json", "w") as f:
    json.dump(eval_results, f, indent=2)

print("Saved: /kaggle/working/phase1_eval_results.json")
print()
print("=" * 60)
print("Copy these into NOTES.md → Phase 1 Results section!")
print("=" * 60)
print(json.dumps(eval_results, indent=2))

## Cell 11 — Save Adapter

Save the LoRA adapter. The `adapter_config.json` file is required by the competition evaluator.

> **Important:** Save the adapter to a Kaggle Dataset (not just notebook output) so it
> persists between sessions. Output files disappear when the session ends!

In [ ]:
# Save LoRA adapter
print(f"Saving adapter to {ADAPTER_DIR}...")
Path(ADAPTER_DIR).mkdir(parents=True, exist_ok=True)

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Verify required files exist
adapter_files = list(Path(ADAPTER_DIR).glob("*"))
print(f"\nAdapter directory contents ({len(adapter_files)} files):")
for f in sorted(adapter_files):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<40} {size_kb:>8.1f} KB")

# Check for adapter_config.json (required by competition)
config_path = Path(ADAPTER_DIR) / "adapter_config.json"
if config_path.exists():
    print(f"\n✅ adapter_config.json found")
    with open(config_path) as f:
        config = json.load(f)
    print(f"   LoRA rank : {config.get('r')}")
    print(f"   Base model: {config.get('base_model_name_or_path')}")
else:
    print("\n❌ adapter_config.json MISSING — submission will fail!")

## Cell 12 — Package Submission

Zip the adapter directory into `submission.zip` as required by the competition.

In [ ]:
SUBMISSION_ZIP = f"{OUTPUT_DIR}/submission.zip"

print(f"Creating {SUBMISSION_ZIP}...")

# zip the adapter directory contents (not the directory itself)
result = subprocess.run(
    ["zip", "-j", SUBMISSION_ZIP] + [str(f) for f in Path(ADAPTER_DIR).glob("*")],
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    print(f"❌ zip failed: {result.stderr}")
else:
    zip_size_mb = Path(SUBMISSION_ZIP).stat().st_size / 1e6
    print(f"✅ submission.zip created ({zip_size_mb:.1f} MB)")

    # Verify zip contents
    verify = subprocess.run(
        ["unzip", "-l", SUBMISSION_ZIP],
        capture_output=True, text=True
    )
    print("\nZip contents:")
    print(verify.stdout)

    if "adapter_config.json" in verify.stdout:
        print("✅ adapter_config.json is in the zip — ready to submit!")
    else:
        print("❌ adapter_config.json NOT found in zip — submission will fail!")

## ✅ Phase 1 Complete!

### What to do next:

1. **Submit** `submission.zip` to Kaggle competition page
2. **Record results** in `NOTES.md` → "Phase 1 Results" section:
   - Leaderboard score
   - Per-category local accuracy (from `phase1_eval_results.json`)
   - Note any surprises (e.g., was roman_numerals already high?)
3. **Update hypotheses** in `NOTES.md` based on results
4. **Move to Phase 2**: Start `phase2_cot_generation.ipynb` to generate CoT reasoning chains

### Phase 2 Preview

The only difference in Phase 2 is the assistant turn:
- **Phase 1**: `<|assistant|>\n\\boxed{answer}`
- **Phase 2**: `<|assistant|>\n{step-by-step reasoning...}\n\n\\boxed{answer}`

The model loading, LoRA setup, and packaging are identical.